In [ ]:
import time
import pubchempy as pcp
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.ensemble import (
    BaggingRegressor,
    GradientBoostingRegressor,
    IsolationForest,
    RandomForestRegressor,
    StackingRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
)
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    RandomizedSearchCV,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor, plot_tree
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

RANDOM_STATE = 100926


def regression_metrics(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "MAPE": mean_absolute_percentage_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


def show_metrics(name, y_true, y_pred):
    values = regression_metrics(y_true, y_pred)
    print(name)
    print(f"  RMSE: {values['RMSE']:.3f}")
    print(f"  MAE:  {values['MAE']:.3f}")
    print(f"  MAPE: {values['MAPE']:.3f}")
    print(f"  R²:   {values['R2']:.3f}")
    return values

## Exploration

In [ ]:
df_raw_train = pd.read_csv("antimicrobial_train_80pct.csv")

In [38]:
df_raw_train

,activity_id,molecule_chembl_id,pref_name,compound_name,canonical_smiles,standard_type,standard_units,assay_chembl_id,rdkit_MaxAbsEStateIndex,rdkit_MaxEStateIndex,...,rdkit_fr_sulfonamd,rdkit_fr_sulfone,rdkit_fr_term_acetylene,rdkit_fr_tetrazole,rdkit_fr_thiazole,rdkit_fr_thiocyan,rdkit_fr_thiophene,rdkit_fr_unbrch_alkane,rdkit_fr_urea,target_log10
0,946920,CHEMBL72286,NaN,NaN,O=C(c1ccccc1O)N1CCc2c(n(Cc3ccc(O)cc3)c3ccccc23)C1,MIC,ug.mL-1,CHEMBL677573,13.081525,13.081525,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.903090
1,379615,CHEMBL134905,NaN,NaN,CC(=O)NC[C@H]1CN(c2ccc3c(c2)CCCCN3C=O)C(=O)O1,MIC,ug.mL-1,CHEMBL677421,12.110715,12.110715,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.301030
2,16571259,CHEMBL3798190,NaN,NaN,CCn1cc(C(=O)O)c(=O)c2cc(F)c(N3CCNCC3)cc21.O=[N...,MIC,ug.mL-1,CHEMBL3801691,14.494909,14.494909,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.892095
3,141787,CHEMBL320039,NaN,NaN,N[C@H]1CN(c2c(F)cc3c(=O)c(C(=O)O)cn(C4CC4)c3c2...,MIC,ug.mL-1,CHEMBL677359,15.030342,15.030342,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.221849
4,551390,CHEMBL88523,NaN,NaN,O=C(OCC1=CC=C(COC(=O)c2cccnc2)SS1)c1cccnc1,MIC,ug.mL-1,CHEMBL648210,11.880338,11.880338,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.301030
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8470,1198295,CHEMBL1160682,NaN,NaN,C[C@@H](O)[C@H]1C(=O)N2C(C(=O)O)=C(Sc3nc(-c4cc...,MIC,ug.mL-1,CHEMBL673662,12.395635,12.395635,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.795880
8471,952203,CHEMBL88982,NaN,NaN,NCCNC(=O)c1cc(-c2c[nH]c3ccc(Br)cc23)nc2ccccc12...,MIC,ug.mL-1,CHEMBL673511,12.647314,12.647314,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.397940
8472,1190294,CHEMBL275692,NaN,NaN,CC(=O)OCC1=C(C(=O)OC(c2ccccc2)c2ccccc2)N2C(=O)...,MIC,ug.mL-1,CHEMBL670512,13.865431,13.865431,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.522879
8473,377863,CHEMBL44431,NaN,NaN,O=C(COc1ccccc1)N[C@@H]1C(=O)N2C(C(=O)O)=C(Cl)C...,MIC,ug.mL-1,CHEMBL673753,12.167056,12.167056,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.806180


In [ ]:
df_raw_test = pd.read_csv("antimicrobial_test_X_20pct.csv")

In [40]:
df_raw_test

,activity_id,molecule_chembl_id,pref_name,compound_name,canonical_smiles,standard_type,standard_units,assay_chembl_id,rdkit_MaxAbsEStateIndex,rdkit_MaxEStateIndex,...,rdkit_fr_sulfide,rdkit_fr_sulfonamd,rdkit_fr_sulfone,rdkit_fr_term_acetylene,rdkit_fr_tetrazole,rdkit_fr_thiazole,rdkit_fr_thiocyan,rdkit_fr_thiophene,rdkit_fr_unbrch_alkane,rdkit_fr_urea
0,849731,CHEMBL121649,NaN,NaN,O=C(Cc1ccccc1)N[C@@H]1C(=O)N2C(C(=O)O)=C(CCO)C...,MIC,ug.mL-1,CHEMBL673564,12.247022,12.247022,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,287078,CHEMBL68208,NaN,NaN,C[C@@H](O)[C@H]1C(=O)N2C(C(=O)[O-])=C(CSC(=S)N...,MIC,ug.mL-1,CHEMBL676864,12.092750,12.092750,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,863129,CHEMBL134718,NaN,NaN,O=C(NC[C@H]1CN(c2ccc(N3CCS(=O)(=O)CC3)c(F)c2)C...,MIC,ug.mL-1,CHEMBL679456,14.677104,14.677104,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,736683,CHEMBL346824,NaN,NaN,O=C(NNc1ccccc1)c1ccc(NC(=S)Nc2ccc(NC(=S)Nc3ccc...,MIC,ug.mL-1,CHEMBL649395,12.406924,12.406924,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1219627,CHEMBL129494,NaN,NaN,CCCCCCCCCCCCOC(=O)c1cc(O)cc(O)c1O,MIC,ug.mL-1,CHEMBL809819,11.825847,11.825847,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2114,5296294,CHEMBL1459,N-DESMETHYL LEVOFLOXACIN,N-DESMETHYL LEVOFLOXACIN,C[C@H]1COc2c(N3CCNCC3)c(F)cc3c(=O)c(C(=O)O)cn1c23,MIC,ug.mL-1,CHEMBL1691259,14.879699,14.879699,...,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2115,162934,CHEMBL284267,NaN,NaN,C=C1/C=C(\C)[C@H](CC)OC(=O)[C@H](C)C(=O)[C@@](...,MIC,ug.mL-1,CHEMBL648394,12.667220,12.667220,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2116,847994,CHEMBL424025,NaN,NaN,CCCCCCCC(=O)c1ccc(O)c(C(=O)Nc2ccc(Br)cc2)c1,MIC,ug.mL-1,CHEMBL638796,12.432809,12.432809,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0
2117,824366,CHEMBL146053,NaN,NaN,Cc1c(N2CCC(O)CC2)ccc2c(=O)c(C(=O)O)cn(C3CC3)c12,MIC,ug.mL-1,CHEMBL674852,12.607102,12.607102,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Check that test has all the same columns except targets

In [ ]:
set(df_raw_train.columns) - set(df_raw_test.columns)

{'target_log10'}

In [42]:
df_raw_train.columns

Index(['activity_id', 'molecule_chembl_id', 'pref_name', 'compound_name',
       'canonical_smiles', 'standard_type', 'standard_units',
       'assay_chembl_id', 'rdkit_MaxAbsEStateIndex', 'rdkit_MaxEStateIndex',
       ...
       'rdkit_fr_sulfonamd', 'rdkit_fr_sulfone', 'rdkit_fr_term_acetylene',
       'rdkit_fr_tetrazole', 'rdkit_fr_thiazole', 'rdkit_fr_thiocyan',
       'rdkit_fr_thiophene', 'rdkit_fr_unbrch_alkane', 'rdkit_fr_urea',
       'target_log10'],
      dtype='str', length=226)

In [ ]:
descriptor_cols = [
    c
    for c in df_raw_train.columns
    if c.startswith("rdkit_") and not c.startswith("rdkit_fr_")
]
df_raw_train.select_dtypes(include=np.number).columns.tolist()
df_raw_train.select_dtypes(exclude=np.number).columns.tolist()
np.isinf(df_raw_train[descriptor_cols]).sum().sort_values(ascending=False)
df_raw_train.columns[df_raw_train.columns.duplicated()].tolist()
(df_raw_train["standard_units"].value_counts().to_dict(),)

({'ug.mL-1': 8475},)

In [ ]:
non_descriptor_cols = [
    column for column in df_raw_train.columns if column not in descriptor_cols
]
non_descriptor_cols

['activity_id',
 'molecule_chembl_id',
 'pref_name',
 'compound_name',
 'canonical_smiles',
 'standard_type',
 'standard_units',
 'assay_chembl_id',
 'rdkit_fr_Al_COO',
 'rdkit_fr_Al_OH',
 'rdkit_fr_Al_OH_noTert',
 'rdkit_fr_ArN',
 'rdkit_fr_Ar_COO',
 'rdkit_fr_Ar_N',
 'rdkit_fr_Ar_NH',
 'rdkit_fr_Ar_OH',
 'rdkit_fr_COO',
 'rdkit_fr_COO2',
 'rdkit_fr_C_O',
 'rdkit_fr_C_O_noCOO',
 'rdkit_fr_C_S',
 'rdkit_fr_HOCCN',
 'rdkit_fr_Imine',
 'rdkit_fr_NH0',
 'rdkit_fr_NH1',
 'rdkit_fr_NH2',
 'rdkit_fr_N_O',
 'rdkit_fr_Ndealkylation1',
 'rdkit_fr_Ndealkylation2',
 'rdkit_fr_Nhpyrrole',
 'rdkit_fr_SH',
 'rdkit_fr_aldehyde',
 'rdkit_fr_alkyl_carbamate',
 'rdkit_fr_alkyl_halide',
 'rdkit_fr_allylic_oxid',
 'rdkit_fr_amide',
 'rdkit_fr_amidine',
 'rdkit_fr_aniline',
 'rdkit_fr_aryl_methyl',
 'rdkit_fr_azide',
 'rdkit_fr_azo',
 'rdkit_fr_barbitur',
 'rdkit_fr_benzene',
 'rdkit_fr_benzodiazepine',
 'rdkit_fr_bicyclic',
 'rdkit_fr_diazo',
 'rdkit_fr_dihydropyridine',
 'rdkit_fr_epoxide',
 'rdkit_fr_es

In [ ]:
with pd.option_context("display.max_rows", None):
    display(df_raw_train.isna().sum())
df_raw_train.groupby("standard_type").agg(
    raw_rows=("molecule_chembl_id", "size"),
    unique_molecules=("molecule_chembl_id", "nunique"),
    molecules_with_repeats=(
        "molecule_chembl_id",
        lambda s: int((s.value_counts() > 1).sum()),
    ),
    maximum_measurements_per_molecule=(
        "molecule_chembl_id",
        lambda s: int(s.value_counts().max()),
    ),
).sort_values("raw_rows", ascending=False)
display(df_raw_train.duplicated().sum())
display(df_raw_train["molecule_chembl_id"].duplicated().sum())
display(df_raw_train["assay_chembl_id"].nunique())
display(df_raw_train["standard_type"].nunique())

activity_id                                 0
molecule_chembl_id                          0
pref_name                                8150
compound_name                            8150
canonical_smiles                            0
standard_type                               0
standard_units                              0
assay_chembl_id                             0
rdkit_MaxAbsEStateIndex                   168
rdkit_MaxEStateIndex                      179
rdkit_MinAbsEStateIndex                   193
rdkit_MinEStateIndex                      147
rdkit_qed                                 180
rdkit_SPS                                 182
rdkit_MolWt                               164
rdkit_HeavyAtomMolWt                      160
rdkit_ExactMolWt                          161
rdkit_NumValenceElectrons                 190
rdkit_NumRadicalElectrons                 166
rdkit_MaxPartialCharge                    190
rdkit_MinPartialCharge                    166
rdkit_MaxAbsPartialCharge         

np.int64(0)

np.int64(0)

1017

1

In [90]:
df_raw_train.loc[df_raw_train[descriptor_cols].isna().any(axis=1), descriptor_cols]

,rdkit_MaxAbsEStateIndex,rdkit_MaxEStateIndex,rdkit_MinAbsEStateIndex,rdkit_MinEStateIndex,rdkit_qed,rdkit_SPS,rdkit_MolWt,rdkit_HeavyAtomMolWt,rdkit_ExactMolWt,rdkit_NumValenceElectrons,...,rdkit_NumRotatableBonds,rdkit_NumSaturatedCarbocycles,rdkit_NumSaturatedHeterocycles,rdkit_NumSaturatedRings,rdkit_NumSpiroAtoms,rdkit_NumUnspecifiedAtomStereoCenters,rdkit_Phi,rdkit_RingCount,rdkit_MolLogP,rdkit_MolMR
0,13.081525,13.081525,0.011976,-0.152602,0.540794,13.400000,398.462,376.286,398.163043,150.0,...,3.0,0.0,0.0,0.0,0.0,0.0,4.432495,5.0,4.2993,115.6981
1,12.110715,12.110715,0.149552,-0.410956,0.847072,20.208333,331.372,310.204,331.153206,128.0,...,NaN,0.0,1.0,1.0,0.0,0.0,4.685374,3.0,1.4470,89.0887
2,14.494909,14.494909,0.100124,-1.300287,0.169941,13.195122,595.884,576.732,595.086596,216.0,...,5.0,0.0,1.0,1.0,0.0,0.0,7.878316,5.0,1.9049,138.8346
3,15.030342,15.030342,0.011741,-1.321205,0.839431,23.370370,391.830,372.678,391.109897,142.0,...,3.0,2.0,1.0,3.0,1.0,0.0,3.435571,5.0,2.7546,100.2617
4,11.880338,11.880338,0.185165,-0.412243,0.549826,13.384615,386.454,372.342,386.039499,132.0,...,6.0,0.0,0.0,0.0,0.0,0.0,6.453500,3.0,3.6532,100.3870
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8470,12.395635,12.395635,0.007995,-1.134669,0.505655,24.892857,417.512,NaN,417.081698,146.0,...,5.0,0.0,1.0,1.0,0.0,0.0,NaN,4.0,2.6381,107.4500
8471,12.647314,12.647314,0.142295,-5.083333,0.314801,11.181818,523.309,505.165,522.051437,172.0,...,4.0,0.0,0.0,0.0,0.0,0.0,7.340465,4.0,4.4674,121.8141
8472,13.865431,13.865431,0.104885,-0.688654,0.243338,18.350000,554.628,528.420,554.162391,202.0,...,8.0,0.0,1.0,1.0,0.0,1.0,6.998034,6.0,4.4085,151.0744
8473,12.167056,12.167056,0.119711,-1.253854,0.752690,22.541667,368.798,355.694,NaN,126.0,...,5.0,0.0,1.0,1.0,0.0,0.0,4.678458,3.0,1.0003,87.4745


In [91]:
TARGET = "target_log10"
MOLECULE_ID = "molecule_chembl_id"
SMILES_COLUMN = "canonical_smiles"

print("Endpoint values:", df_raw_train["standard_type"].value_counts().to_dict())
print("Units:", df_raw_train["standard_units"].value_counts().to_dict())
print("Unique molecules:", df_raw_train[MOLECULE_ID].nunique())
print("Repeated raw rows:", df_raw_train.duplicated().sum())

Endpoint values: {'MIC': 8475}
Units: {'ug.mL-1': 8475}
Unique molecules: 8475
Repeated raw rows: 0


In [92]:
print("Endpoint values:", df_raw_test["standard_type"].value_counts().to_dict())
print("Units:", df_raw_test["standard_units"].value_counts().to_dict())

Endpoint values: {'MIC': 2119}
Units: {'ug.mL-1': 2119}


In [ ]:
target_distribution_fig = px.histogram(
    df_raw_train,
    x=TARGET,
    nbins=35,
    color="standard_type",
    marginal="box",
    title="Raw assay target distribution by endpoint",
    labels={TARGET: "target log10 value", "count": "Raw assay rows"},
)
target_distribution_fig.update_layout(barmode="overlay")
target_distribution_fig.update_traces(opacity=0.70)
target_distribution_fig.show()


In [94]:
endpoint_measurement_counts = df_raw_train[MOLECULE_ID].value_counts()
endpoint_measurement_counts.nunique()

1

In [ ]:
df_raw_train[descriptor_cols] = df_raw_train[descriptor_cols].replace(
    [np.inf, -np.inf], np.nan
)
constant = [c for c in descriptor_cols if df_raw_train[c].nunique(dropna=True) <= 1]
float32_limit = np.finfo(np.float32).max

too_large = [
    c for c in descriptor_cols if df_raw_train[c].abs().max(skipna=True) > float32_limit
]

print(too_large)
print(constant)

['rdkit_Ipc']
['rdkit_NumRadicalElectrons', 'rdkit_SMR_VSA8', 'rdkit_SlogP_VSA9']


In [ ]:
feature_cols = [c for c in descriptor_cols if c not in set(constant + too_large)]

molecule_df = df_raw_train.dropna(subset=[TARGET]).reset_index(drop=True)
X = molecule_df[feature_cols].copy()
y = molecule_df[TARGET].copy()
X


,rdkit_MaxAbsEStateIndex,rdkit_MaxEStateIndex,rdkit_MinAbsEStateIndex,rdkit_MinEStateIndex,rdkit_qed,rdkit_SPS,rdkit_MolWt,rdkit_HeavyAtomMolWt,rdkit_ExactMolWt,rdkit_NumValenceElectrons,...,rdkit_NumRotatableBonds,rdkit_NumSaturatedCarbocycles,rdkit_NumSaturatedHeterocycles,rdkit_NumSaturatedRings,rdkit_NumSpiroAtoms,rdkit_NumUnspecifiedAtomStereoCenters,rdkit_Phi,rdkit_RingCount,rdkit_MolLogP,rdkit_MolMR
0,13.081525,13.081525,0.011976,-0.152602,0.540794,13.400000,398.462,376.286,398.163043,150.0,...,3.0,0.0,0.0,0.0,0.0,0.0,4.432495,5.0,4.2993,115.6981
1,12.110715,12.110715,0.149552,-0.410956,0.847072,20.208333,331.372,310.204,331.153206,128.0,...,NaN,0.0,1.0,1.0,0.0,0.0,4.685374,3.0,1.4470,89.0887
2,14.494909,14.494909,0.100124,-1.300287,0.169941,13.195122,595.884,576.732,595.086596,216.0,...,5.0,0.0,1.0,1.0,0.0,0.0,7.878316,5.0,1.9049,138.8346
3,15.030342,15.030342,0.011741,-1.321205,0.839431,23.370370,391.830,372.678,391.109897,142.0,...,3.0,2.0,1.0,3.0,1.0,0.0,3.435571,5.0,2.7546,100.2617
4,11.880338,11.880338,0.185165,-0.412243,0.549826,13.384615,386.454,372.342,386.039499,132.0,...,6.0,0.0,0.0,0.0,0.0,0.0,6.453500,3.0,3.6532,100.3870
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8470,12.395635,12.395635,0.007995,-1.134669,0.505655,24.892857,417.512,NaN,417.081698,146.0,...,5.0,0.0,1.0,1.0,0.0,0.0,NaN,4.0,2.6381,107.4500
8471,12.647314,12.647314,0.142295,-5.083333,0.314801,11.181818,523.309,505.165,522.051437,172.0,...,4.0,0.0,0.0,0.0,0.0,0.0,7.340465,4.0,4.4674,121.8141
8472,13.865431,13.865431,0.104885,-0.688654,0.243338,18.350000,554.628,528.420,554.162391,202.0,...,8.0,0.0,1.0,1.0,0.0,1.0,6.998034,6.0,4.4085,151.0744
8473,12.167056,12.167056,0.119711,-1.253854,0.752690,22.541667,368.798,355.694,NaN,126.0,...,5.0,0.0,1.0,1.0,0.0,0.0,4.678458,3.0,1.0003,87.4745


In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

TSNE_SAMPLE_SIZE = min(1500, len(X))
tsne_indices = molecule_df.sample(n=TSNE_SAMPLE_SIZE, random_state=RANDOM_STATE).index
tsne_imputer = SimpleImputer(strategy="median")
X_tsne = tsne_imputer.fit_transform(X.loc[tsne_indices])
X_tsne = StandardScaler().fit_transform(X_tsne)
tsne_embedding = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=RANDOM_STATE,
).fit_transform(X_tsne)

tsne_plot_df = molecule_df.loc[tsne_indices, [MOLECULE_ID, TARGET]].copy()
tsne_plot_df["TSNE1"] = tsne_embedding[:, 0]
tsne_plot_df["TSNE2"] = tsne_embedding[:, 1]

tsne_fig = px.scatter(
    tsne_plot_df,
    x="TSNE1",
    y="TSNE2",
    color=TARGET,
    hover_name=MOLECULE_ID,
    hover_data=[TARGET],
    color_continuous_scale="Viridis",
    title=f"t-SNE of RDKit descriptors, colored by {TARGET}",
)
tsne_fig.update_traces(marker={"size": 7, "opacity": 0.75})
tsne_fig.show()


In [ ]:
target_col = TARGET  # Replace with your actual target column name

# 1. Select only numeric columns (correlation requires numbers)
numeric_df = molecule_df[feature_cols + [target_col]].select_dtypes(include=["number"])

# 2. Calculate correlation with the target and sort from highest to lowest
correlations = numeric_df.corr()[target_col].sort_values(ascending=False)

# 3. Display the results
with pd.option_context("display.max_rows", None):
    display(correlations)

target_log10                             1.000000
rdkit_PEOE_VSA6                          0.189076
rdkit_EState_VSA6                        0.135702
rdkit_VSA_EState6                        0.131612
rdkit_MinAbsEStateIndex                  0.124454
rdkit_MinPartialCharge                   0.120774
rdkit_SlogP_VSA3                         0.112498
rdkit_EState_VSA9                        0.110494
rdkit_BalabanJ                           0.109302
rdkit_SlogP_VSA6                         0.097301
rdkit_BCUT2D_CHGLO                       0.095144
rdkit_Kappa3                             0.087373
rdkit_SlogP_VSA7                         0.084975
rdkit_Phi                                0.077094
rdkit_EState_VSA7                        0.076215
rdkit_BCUT2D_LOGPLOW                     0.068825
rdkit_NumUnspecifiedAtomStereoCenters    0.063354
rdkit_Kappa2                             0.062884
rdkit_EState_VSA8                        0.061753
rdkit_NumAtomStereoCenters               0.060926


## Decision tree

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)


In [ ]:
import optuna


def objective_rf(trial):
    model = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=trial.suggest_int("n_estimators", 10, 350),
                    max_depth=trial.suggest_int("max_depth", 2, 50),
                    min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 30),
                    max_features=trial.suggest_categorical(
                        "max_features", ["sqrt", 0.01, 0.5, 0.8, 1]
                    ),
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    )
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=3,
        scoring="neg_mean_absolute_percentage_error",
        n_jobs=-1,
    )
    return -scores.mean()


study_rf = optuna.create_study(direction="minimize")
study_rf.optimize(objective_rf, n_trials=50, show_progress_bar=False)
print("Best Optuna RMSE:", study_rf.best_value)
print("Best Optuna parameters:", study_rf.best_params)

[I 2026-10-10 13:38:21,831] A new study created in memory with name: no-name-904299c4-6520-4b6e-9bb3-2e4d50323c30
[I 2026-10-10 13:38:22,348] Trial 0 finished with value: 100564490664386.14 and parameters: {'n_estimators': 111, 'max_depth': 6, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 100564490664386.14.
[I 2026-10-10 13:38:22,761] Trial 1 finished with value: 96343067356433.19 and parameters: {'n_estimators': 154, 'max_depth': 23, 'min_samples_leaf': 17, 'max_features': 0.01}. Best is trial 1 with value: 96343067356433.19.
[I 2026-10-10 13:38:23,947] Trial 2 finished with value: 87225024427731.77 and parameters: {'n_estimators': 231, 'max_depth': 37, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 2 with value: 87225024427731.77.
[I 2026-10-10 13:38:24,461] Trial 3 finished with value: 92054562087869.19 and parameters: {'n_estimators': 342, 'max_depth': 44, 'min_samples_leaf': 6, 'max_features': 1}. Best is trial 2 with value: 8722502442

Best Optuna RMSE: 79376215106115.73
Best Optuna parameters: {'n_estimators': 140, 'max_depth': 31, 'min_samples_leaf': 1, 'max_features': 0.8}


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error

# 1. Re-build the model using the best hyperparameters found by Optuna
best_params = study_rf.best_params

best_model = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            RandomForestRegressor(
                **best_params,  # Unpacks the dictionary of best parameters
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

# 2. Fit the model on the full training data
best_model.fit(X_train, y_train)

# 3. Make predictions on your test (or validation) set
# (Assuming you have X_test and y_test defined)
y_pred = best_model.predict(X_test)

# 4. Calculate MAPE
best_mape = mean_absolute_percentage_error(y_test, y_pred)

print(f"Best Model MAPE: {best_mape:.3f}")

# Optional: If you want to use the show_metrics function from your previous step
# show_metrics("Optuna Best Model", y_test, y_pred)

Best Model MAPE: 88421039143831.938


In [ ]:
def build_optuna_stack(params):
    return StackingRegressor(
        estimators=[
            (
                "tree",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        (
                            "model",
                            DecisionTreeRegressor(
                                max_depth=params["tree_max_depth"],
                                min_samples_leaf=params["tree_min_samples_leaf"],
                                ccp_alpha=params["tree_ccp_alpha"],
                                random_state=RANDOM_STATE,
                            ),
                        ),
                    ]
                ),
            ),
            (
                "rf",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        (
                            "model",
                            RandomForestRegressor(
                                n_estimators=params["rf_n_estimators"],
                                max_depth=params["rf_max_depth"],
                                max_features=params["rf_max_features"],
                                min_samples_leaf=params["rf_min_samples_leaf"],
                                random_state=RANDOM_STATE,
                                n_jobs=1,
                            ),
                        ),
                    ]
                ),
            ),
            (
                "gb",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        (
                            "model",
                            GradientBoostingRegressor(
                                n_estimators=params["gb_n_estimators"],
                                learning_rate=params["gb_learning_rate"],
                                max_depth=params["gb_max_depth"],
                                min_samples_leaf=params["gb_min_samples_leaf"],
                                subsample=params["gb_subsample"],
                                random_state=RANDOM_STATE,
                            ),
                        ),
                    ]
                ),
            ),
            (
                "lgb",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        (
                            "model",
                            LGBMRegressor(
                                n_estimators=params["lgb_n_estimators"],
                                learning_rate=params["lgb_learning_rate"],
                                num_leaves=params["lgb_num_leaves"],
                                max_depth=params["lgb_max_depth"],
                                min_child_samples=params["lgb_min_child_samples"],
                                subsample=params["lgb_subsample"],
                                colsample_bytree=params["lgb_colsample_bytree"],
                                random_state=RANDOM_STATE,
                                verbosity=-1,
                                n_jobs=1,
                            ),
                        ),
                    ]
                ),
            ),
            (
                "xgb",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        (
                            "model",
                            XGBRegressor(
                                n_estimators=params["xgb_n_estimators"],
                                learning_rate=params["xgb_learning_rate"],
                                max_depth=params["xgb_max_depth"],
                                min_child_weight=params["xgb_min_child_weight"],
                                subsample=params["xgb_subsample"],
                                colsample_bytree=params["xgb_colsample_bytree"],
                                reg_lambda=params["xgb_reg_lambda"],
                                objective="reg:squarederror",
                                random_state=RANDOM_STATE,
                                n_jobs=1,
                            ),
                        ),
                    ]
                ),
            ),
        ],
        final_estimator=Ridge(alpha=params["ridge_alpha"]),
        cv=3,
        n_jobs=1,
    )


def optuna_stack_objective(trial):
    params = {
        "tree_max_depth": trial.suggest_int("tree_max_depth", 2, 19),
        "tree_min_samples_leaf": trial.suggest_int("tree_min_samples_leaf", 1, 60),
        "tree_ccp_alpha": trial.suggest_float("tree_ccp_alpha", 1e-6, 0.2, log=True),
        "rf_n_estimators": trial.suggest_int("rf_n_estimators", 20, 400),
        "rf_max_depth": trial.suggest_int("rf_max_depth", 3, 20),
        "rf_max_features": trial.suggest_categorical(
            "rf_max_features", ["sqrt", 0.25, 0.5, 0.8]
        ),
        "rf_min_samples_leaf": trial.suggest_int("rf_min_samples_leaf", 1, 15),
        "gb_n_estimators": trial.suggest_int("gb_n_estimators", 30, 400),
        "gb_learning_rate": trial.suggest_float(
            "gb_learning_rate", 0.01, 0.3, log=True
        ),
        "gb_max_depth": trial.suggest_int("gb_max_depth", 1, 7),
        "gb_min_samples_leaf": trial.suggest_int("gb_min_samples_leaf", 2, 50),
        "gb_subsample": trial.suggest_float("gb_subsample", 0.2, 1.0),
        "lgb_n_estimators": trial.suggest_int("lgb_n_estimators", 20, 400),
        "lgb_learning_rate": trial.suggest_float(
            "lgb_learning_rate", 0.01, 0.2, log=True
        ),
        "lgb_num_leaves": trial.suggest_int("lgb_num_leaves", 5, 63),
        "lgb_max_depth": trial.suggest_categorical("lgb_max_depth", [-1, 3, 5, 8, 12]),
        "lgb_min_child_samples": trial.suggest_int("lgb_min_child_samples", 5, 30),
        "lgb_subsample": trial.suggest_float("lgb_subsample", 0.1, 1.0),
        "lgb_colsample_bytree": trial.suggest_float("lgb_colsample_bytree", 0.2, 1.0),
        "xgb_n_estimators": trial.suggest_int("xgb_n_estimators", 20, 400),
        "xgb_learning_rate": trial.suggest_float(
            "xgb_learning_rate", 0.001, 0.3, log=True
        ),
        "xgb_max_depth": trial.suggest_int("xgb_max_depth", 2, 14),
        "xgb_min_child_weight": trial.suggest_int("xgb_min_child_weight", 1, 10),
        "xgb_subsample": trial.suggest_float("xgb_subsample", 0.3, 1.0),
        "xgb_colsample_bytree": trial.suggest_float("xgb_colsample_bytree", 0.3, 1.0),
        "xgb_reg_lambda": trial.suggest_float("xgb_reg_lambda", 1e-3, 10.0, log=True),
        "ridge_alpha": trial.suggest_float("ridge_alpha", 1e-4, 100.0, log=True),
    }
    model = build_optuna_stack(params)

    # Changed scoring metric to MAPE
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=3,
        scoring="neg_mean_absolute_percentage_error",
        n_jobs=-1,
    )

    # Changed user attribute name to reflect MAPE
    trial.set_user_attr("cv_mape_sd", float(scores.std()))
    return -scores.mean()


STACKING_OPTUNA_TRIALS = 60

# 1. Change WARNING to INFO so it prints the result of each trial
optuna.logging.set_verbosity(optuna.logging.INFO)

stack_study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)

# 2. Change show_progress_bar to True
stack_study.optimize(
    optuna_stack_objective,
    n_trials=STACKING_OPTUNA_TRIALS,
    show_progress_bar=True,
)

print("Best tuned stacking CV MAPE:", round(stack_study.best_value, 5))

[I 2026-10-10 14:33:23,207] A new study created in memory with name: no-name-085cb1fb-a3c0-40e8-ad68-9c6b31423882


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-10-10 14:34:41,391] Trial 0 finished with value: 83519213428178.64 and parameters: {'tree_max_depth': 7, 'tree_min_samples_leaf': 58, 'tree_ccp_alpha': 3.931012922319056e-06, 'rf_n_estimators': 136, 'rf_max_depth': 18, 'rf_max_features': 0.5, 'rf_min_samples_leaf': 10, 'gb_n_estimators': 343, 'gb_learning_rate': 0.05936586244163403, 'gb_max_depth': 3, 'gb_min_samples_leaf': 28, 'gb_subsample': 0.7929041905505685, 'lgb_n_estimators': 286, 'lgb_learning_rate': 0.0478743704266611, 'lgb_num_leaves': 12, 'lgb_max_depth': 8, 'lgb_min_child_samples': 29, 'lgb_subsample': 0.8678123491004716, 'lgb_colsample_bytree': 0.5911964253046155, 'xgb_n_estimators': 325, 'xgb_learning_rate': 0.06824231527897888, 'xgb_max_depth': 10, 'xgb_min_child_weight': 10, 'xgb_subsample': 0.5545813589260403, 'xgb_colsample_bytree': 0.5145532110820243, 'xgb_reg_lambda': 0.23086424549816795, 'ridge_alpha': 0.09213069087902985}. Best is trial 0 with value: 83519213428178.64.
[I 2026-10-10 14:36:09,764] Trial 1 f

KeyboardInterrupt: 

In [ ]:
import plotly.express as px
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 1. Build a pipeline to impute, scale, and extract the top 2 PCs
pca_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=20, random_state=RANDOM_STATE)),
    ]
)

# 2. Process the training data
components = pca_pipeline.fit_transform(X_train)

# Extract the fitted PCA model from the pipeline to get the variance ratios
fitted_pca = pca_pipeline.named_steps["pca"]

# 3. Combine the components and your target variable into a single DataFrame
pca_df = pd.DataFrame(data=components[:, :2], columns=["PC1", "PC2"])
pca_df["Target"] = y_train.values  # .values prevents pandas index-mismatch errors

# Calculate total variance explained by these 2 components
explained_var = fitted_pca.explained_variance_ratio_.sum() * 100

# 4. Plot using Plotly Express
fig = px.scatter(
    pca_df,
    x="PC1",
    y="PC2",
    color="Target",
    title=f"PCA 2D Projection (Captures {explained_var:.1f}% of Total Variance)",
    color_continuous_scale="Plasma",
    opacity=0.7,
)

# Update layout for cleaner presentation
fig.update_layout(
    xaxis_title=f"Principal Component 1 ({fitted_pca.explained_variance_ratio_[0] * 100:.1f}%)",
    yaxis_title=f"Principal Component 2 ({fitted_pca.explained_variance_ratio_[1] * 100:.1f}%)",
    template="plotly_white",
)

fig.show()

In [ ]:
import numpy as np

# 1. Start with completely clean training data
train_mask = y_train.values.ravel() != 0.0
X_train_clean = X_train[train_mask]
y_train_clean = y_train.values.ravel()[train_mask]


# 2. Update the Objective Function (No PCA, Clean Data)
def optuna_stack_objective(trial):
    # Let Optuna focus on the heavy hitters for this task
    params = {
        "lgb_n_estimators": trial.suggest_int("lgb_n_estimators", 50, 400),
        "lgb_learning_rate": trial.suggest_float(
            "lgb_learning_rate", 0.01, 0.2, log=True
        ),
        "lgb_num_leaves": trial.suggest_int("lgb_num_leaves", 10, 60),
        "xgb_n_estimators": trial.suggest_int("xgb_n_estimators", 50, 400),
        "xgb_learning_rate": trial.suggest_float(
            "xgb_learning_rate", 0.01, 0.2, log=True
        ),
        "xgb_max_depth": trial.suggest_int("xgb_max_depth", 3, 10),
        "rf_n_estimators": trial.suggest_int("rf_n_estimators", 50, 200),
        "rf_max_depth": trial.suggest_int("rf_max_depth", 5, 20),
    }

    # Simplified stack of the 3 best models (all using absolute error natively)
    stack_model = StackingRegressor(
        estimators=[
            (
                "lgb",
                LGBMRegressor(
                    n_estimators=params["lgb_n_estimators"],
                    learning_rate=params["lgb_learning_rate"],
                    num_leaves=params["lgb_num_leaves"],
                    objective="mape",  # Native MAPE
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                    verbosity=-1,
                ),
            ),
            (
                "xgb",
                XGBRegressor(
                    n_estimators=params["xgb_n_estimators"],
                    learning_rate=params["xgb_learning_rate"],
                    max_depth=params["xgb_max_depth"],
                    objective="reg:absoluteerror",  # Native MAE
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                ),
            ),
            (
                "rf",
                RandomForestRegressor(
                    n_estimators=params["rf_n_estimators"],
                    max_depth=params["rf_max_depth"],
                    criterion="absolute_error",  # Native MAE
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                ),
            ),
        ],
        final_estimator=GradientBoostingRegressor(
            loss="absolute_error", random_state=RANDOM_STATE
        ),
        cv=3,
        n_jobs=1,
    )

    # Master Pipeline (Impute + Scale -> directly to Stack)
    master_pipeline = Pipeline(
        [("imputer", SimpleImputer(strategy="median")), ("stack", stack_model)]
    )

    # Cross-validate on the CLEAN data
    scores = cross_val_score(
        master_pipeline,
        X_train_clean,
        y_train_clean,
        cv=3,
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
    )

    return -scores.mean()


# 3. Run the study (30 trials should be enough for this simplified space)
study = optuna.create_study(
    direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study.optimize(optuna_stack_objective, n_trials=30, show_progress_bar=True)

[I 2026-10-10 16:27:33,629] A new study created in memory with name: no-name-7ab78a27-c77f-449c-ac0b-5bfaac6d88e0


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-10-10 16:30:05,819] Trial 0 finished with value: 0.5934862851190554 and parameters: {'lgb_n_estimators': 162, 'lgb_learning_rate': 0.17950110417592738, 'lgb_num_leaves': 15, 'xgb_n_estimators': 157, 'xgb_learning_rate': 0.12199848016418945, 'xgb_max_depth': 4, 'rf_n_estimators': 107, 'rf_max_depth': 13}. Best is trial 0 with value: 0.5934862851190554.
[I 2026-10-10 16:34:21,725] Trial 1 finished with value: 0.5779423244670349 and parameters: {'lgb_n_estimators': 146, 'lgb_learning_rate': 0.06132616337626126, 'lgb_num_leaves': 53, 'xgb_n_estimators': 233, 'xgb_learning_rate': 0.027248016989342153, 'xgb_max_depth': 7, 'rf_n_estimators': 161, 'rf_max_depth': 16}. Best is trial 1 with value: 0.5779423244670349.
[I 2026-10-10 16:35:49,926] Trial 2 finished with value: 0.5923440745297025 and parameters: {'lgb_n_estimators': 233, 'lgb_learning_rate': 0.0148848749645285, 'lgb_num_leaves': 15, 'xgb_n_estimators': 146, 'xgb_learning_rate': 0.01308931691200134, 'xgb_max_depth': 6, 'rf_n_e

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error
import numpy as np

# 1. Extract the best hyperparameters found by Optuna
best_params = stack_study.best_params

# 2. Rebuild the base stacking model using your existing function
best_stack = build_optuna_stack(best_params)

# 3. Rebuild the complete Master Pipeline (Scaler removed)
final_pipeline = Pipeline(
    [("imputer", SimpleImputer(strategy="median")), ("stack", best_stack)]
)

# 4. TRAIN FIX: Ensure you train ONLY on the clean training data
train_mask = y_train.values.ravel() != 0.0
X_train_clean = X_train[train_mask]
y_train_clean = y_train.values.ravel()[train_mask]

final_pipeline.fit(X_train_clean, y_train_clean)

# 5. Generate predictions for the test set
y_pred = final_pipeline.predict(X_test)

# 6. EVALUATION FIX: Filter out the exact 0.0s from the test set before scoring
y_test_arr = y_test.values.ravel()
y_pred_arr = y_pred.ravel()

# Create a mask for valid rows (where target is not exactly 0.0)
valid_mask = y_test_arr != 0.0

y_test_valid = y_test_arr[valid_mask]
y_pred_valid = y_pred_arr[valid_mask]

# 7. Calculate MAPE using sklearn on the valid rows only
final_mape = mean_absolute_percentage_error(y_test_valid, y_pred_valid)

print(f"Removed {(~valid_mask).sum()} pseudo-NA targets from test evaluation.")
print(f"Final Test MAPE: {final_mape:.2f}%")

Removed 58 pseudo-NA targets from test evaluation.
Final Test MAPE: 0.95%


In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_percentage_error

# 1. Purge the pseudo-NAs from the TRAINING data before fitting
train_mask = y_train.values.ravel() != 0.0
X_train_clean = X_train[train_mask]
y_train_clean = y_train.values.ravel()[train_mask]

print(f"Removed {(~train_mask).sum()} poisoned rows from training data.")

# 2. Retrain the entire master pipeline on the CLEANED data
final_pipeline.fit(X_train_clean, y_train_clean)

# 3. Purge the pseudo-NAs from the TEST data
test_mask = y_test.values.ravel() != 0.0
X_test_clean = X_test[test_mask]
y_test_clean = y_test.values.ravel()[test_mask]

# 4. Generate fresh predictions
y_pred_clean = final_pipeline.predict(X_test_clean)

# 5. Evaluate Log-Scale MAPE (Just in case)
log_mape = np.mean(np.abs((y_test_clean - y_pred_clean) / y_test_clean)) * 100
print(f"Log-Scale MAPE: {log_mape:.2f}%")

# 6. Evaluate Original-Scale MAPE (The standard method for cheminformatics)
# 10**x reverses the log10 transformation to evaluate the true biological percentage error
y_test_original = 10**y_test_clean
y_pred_original = 10**y_pred_clean

true_biological_mape = (
    mean_absolute_percentage_error(y_test_original, y_pred_original) * 100
)
print(f"Original-Scale MAPE: {true_biological_mape:.2f}%")

Removed 221 poisoned rows from training data.
Log-Scale MAPE: 112.05%
Original-Scale MAPE: 3083.20%


In [ ]:
(y_train == 0).sum()

np.int64(221)

In [ ]:
(y_test == 0).sum()

np.int64(58)

In [ ]:
import pandas as pd
import plotly.express as px
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 1. Build the pipeline for 10 components
pca_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=10, random_state=RANDOM_STATE)),
    ]
)

# 2. Transform the training data into Principal Components
pcs = pca_pipeline.fit_transform(X_train)

# 3. Create a DataFrame for the PCs
pc_columns = [f"PC{i + 1}" for i in range(10)]
pc_df = pd.DataFrame(data=pcs, columns=pc_columns)

# 4. Add the target variable (using .values to avoid pandas index mismatches)
pc_df["Target"] = y_train.values

# 5. Calculate Pearson correlation with the target
correlations = pc_df.corr()["Target"].drop("Target").sort_values(ascending=False)

print("Correlation with Target:")
print(correlations)

# 6. Visualize with Plotly
fig = px.bar(
    x=correlations.index,
    y=correlations.values,
    title="Correlation of Top 10 PCs with Target",
    labels={"x": "Principal Component", "y": "Pearson Correlation"},
    color=correlations.values,
    color_continuous_scale="RdBu",  # Red for negative, Blue for positive
    template="plotly_white",
)

# Fix axis ranges for clarity (-1 to 1 is the max possible correlation)
fig.update_yaxes(range=[-1, 1])
fig.show()

Correlation with Target:
PC3     0.174827
PC8     0.160703
PC10    0.023683
PC6     0.022755
PC1     0.009817
PC9    -0.045702
PC2    -0.121786
PC7    -0.130235
PC5    -0.138926
PC4    -0.140281
Name: Target, dtype: float64


In [ ]:
df_raw_train["target_log10"].value_counts()

target_log10
2.000000    588
2.107210    530
1.698970    416
0.301030    352
0.903090    330
           ... 
3.273233      1
0.518514      1
3.311330      1
2.510545      1
2.057666      1
Name: count, Length: 385, dtype: int64

In [ ]:
# Define the most common pseudo-NA markers used in bioinformatics datasets
pseudo_na_markers = [999, 9999, 99]

# 1. Count how many times these markers appear in each column
pseudo_counts = df_raw_train.isin(pseudo_na_markers).sum()

# 2. Filter to show only the columns that actually contain them
columns_with_pseudos = pseudo_counts[pseudo_counts > 0].sort_values(ascending=False)

if len(columns_with_pseudos) > 0:
    print(f"Found pseudo-NAs in {len(columns_with_pseudos)} columns:\n")
    print(columns_with_pseudos)

    # 3. Optional: See exactly WHICH fake values are being used in a specific column
    # Replace 'your_top_column' with the first name printed above
    # print("\nValue breakdown for top column:")
    # print(df_raw[df_raw['your_top_column'].isin(pseudo_na_markers)]['your_top_column'].value_counts())
else:
    print("No standard extreme pseudo-NAs (-999, 999, etc.) found in the features.")

Found pseudo-NAs in 1 columns:

rdkit_HeavyAtomCount    11
dtype: int64
